<a href="https://colab.research.google.com/github/springboardmentor787-stack/Company-Internal-Chatbot-with-Role-Based-Access-Control-RBAC---Group-1/blob/Rithika-Damacharla/milestone_2(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
langchain \
langchain-community \
sentence-transformers \
chromadb \
pandas \
numpy \
tqdm \
unstructured \
markdown


In [2]:
!git clone https://github.com/springboardmentor441p-coderr/Fintech-data.git


fatal: destination path 'Fintech-data' already exists and is not an empty directory.


In [3]:
!ls Fintech-data


engineering  Finance  general  HR  marketing


In [4]:
# Step 1: Role → Departments mapping
ROLE_TO_DEPARTMENTS = {
    "Finance": ["finance", "general"],
    "Marketing": ["marketing", "general"],
    "HR": ["hr", "general"],
    "Engineering": ["engineering", "general"],
    "Employees": ["general"],
    "C-Level": ["finance", "marketing", "hr", "engineering", "general"]
}

# Step 1b: Reverse mapping for printing allowed roles
DEPARTMENT_ACCESS = {}
for role, departments in ROLE_TO_DEPARTMENTS.items():
    for dept in departments:
        DEPARTMENT_ACCESS.setdefault(dept, []).append(role)


In [5]:
def detect_doc_department(path, text=""):
    p = path.lower()

    if "finance" in p:
        return "finance"

    if "marketing" in p:
        return "marketing"

    if "hr" in p:
        return "hr"

    if "engineering" in p or "tech" in p:
        return "engineering"

    if "general" in p:
        return "general"

    return "general"


In [11]:
import os
documents = []

base_path = "Fintech-data"

for root, dirs, files in os.walk(base_path):
    for file in files:
        file_path = os.path.join(root, file)

        department = os.path.basename(root).lower()

        if file.endswith(".md"):
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

        elif file.endswith(".csv"):
            df = pd.read_csv(file_path)
            text = df.to_string(index=False)

        else:
            continue

        documents.append({
            "text": text,
            "source": file,
            "department": department
        })


In [12]:
def detect_department(query):
    q = query.lower()

    # Finance
    if any(word in q for word in ["revenue", "profit", "loss", "budget", "expense"]):
        return "finance"

    # Marketing
    if any(word in q for word in ["campaign", "marketing", "brand", "seo", "ads"]):
        return "marketing"

    # Engineering
    if any(word in q for word in ["api", "backend", "frontend", "architecture", "system"]):
        return "engineering"

    # HR — ONLY sensitive HR terms
    if any(word in q for word in ["salary", "reimbursement", "payroll", "leave"]):
        return "hr"

    # Everything else
    return "general"


In [13]:
def validate_role_access(user_role, query_department):
    allowed_departments = ROLE_TO_DEPARTMENTS.get(user_role, [])
    return query_department in allowed_departments


In [14]:
set(doc["department"] for doc in documents)

{'engineering', 'finance', 'general', 'hr', 'marketing'}

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = []

for doc in documents:
    split_texts = text_splitter.split_text(doc["text"])

    for i, chunk in enumerate(split_texts):
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "department": doc["department"],
            "chunk_id": i
        })

print("Total chunks created:", len(chunks))

Total chunks created: 350


milestone 2


In [16]:
from chromadb.utils import embedding_functions
import chromadb

# Initialize client
chroma_client = chromadb.Client()

# Create collection with embedding function
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.create_collection(
    name="company_docs",
    embedding_function=embedding_function,
    get_or_create=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [17]:
texts = [c["text"] for c in chunks]
metadatas = [
    {
        "source": c.get("source", "unknown"),
        "department": c.get("department", "general").lower()
    }
    for c in chunks
]

ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print("✅ Vector DB populated successfully")

✅ Vector DB populated successfully


In [18]:
def semantic_search(query, n_results=5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results


In [19]:
def role_based_search(query, user_role, n_results=5):

    allowed_departments = ROLE_TO_DEPARTMENTS[user_role]

    results = collection.query(
        query_texts=[query],
        n_results=15
    )

    filtered_docs = []

    for doc, meta in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        if meta["department"] in allowed_departments:
            filtered_docs.append((doc, meta))

    return filtered_docs[:n_results]

In [54]:
def role_based_search_results(query, user_role, top_k=5):

    print("======================================")
    print("USER QUERY :", query)
    print("USER ROLE  :", user_role)
    print("======================================")

    # -------------------------------
    # Validate role
    # -------------------------------
    if user_role not in ROLE_TO_DEPARTMENTS:
        print("❌ Invalid role.")
        return

    allowed_departments = ROLE_TO_DEPARTMENTS[user_role]

    # =========================================================
    # ✅ C-LEVEL BLOCK — PRINT ALL DEPARTMENTS
    # =========================================================
    if user_role == "C-Level":

        expected_departments = ["finance", "hr", "marketing", "engineering", "general"]

        results = collection.query(
            query_texts=[query],
            n_results=50
        )

        dept_docs = {}

        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            dept = meta.get("department", "general")
            if dept not in dept_docs:
                dept_docs[dept] = doc

        print("\n✅ C-LEVEL ACCESS — ALL DEPARTMENTS\n")

        for i, dept in enumerate(expected_departments, 1):
            print(f"RESULT {i}")
            print("-------------------------------")
            print("Department :", dept)
            print("User Role  : C-Level")
            print("Access     : TRUE ✅")

            if dept in dept_docs:
                print("\nContent Preview:")
                print(dept_docs[dept][:300])
            else:
                print("\nContent Preview:")
                print("⚠️ No document available for this department.")

            print("\n======================================")

        return

    # =========================================================
    # ✅ NON C-LEVEL USERS
    # =========================================================

    query_department = detect_department(query)
    print("QUERY DEPARTMENT:", query_department)

    # 🔒 role–query connection
    if query_department not in allowed_departments:
        print("❌ ACCESS DENIED — Query not permitted for this role.")
        return

    # Semantic search
    results = collection.query(
        query_texts=[query],
        n_results=25
    )

    filtered_docs = []

    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        dept = meta.get("department", "general")
        if dept in allowed_departments:
            filtered_docs.append((doc, meta))

    if not filtered_docs:
        print("❌ No authorized documents found.")
        return

    for i, (doc, meta) in enumerate(filtered_docs[:top_k], 1):
        print(f"\nRESULT {i}")
        print("-------------------------------")
        print("Returned Department :", meta["department"])
        print("User Role           :", user_role)
        print("Allowed Departments :", allowed_departments)
        print("Access Granted      : TRUE ✅")
        print("\nContent Preview:")
        print(doc[:300])
        print("\n======================================")


In [55]:
role_based_search_results(
    query="quarterly revenue and expenses",
    user_role="HR"
)

USER QUERY : quarterly revenue and expenses
USER ROLE  : HR
QUERY DEPARTMENT: finance
❌ ACCESS DENIED — Query not permitted for this role.


In [56]:
role_based_search_results(
    "system architecture and company policies",
    "Engineering"
)


USER QUERY : system architecture and company policies
USER ROLE  : Engineering
QUERY DEPARTMENT: engineering

RESULT 1
-------------------------------
Returned Department : engineering
User Role           : Engineering
Allowed Departments : ['engineering', 'general']
Access Granted      : TRUE ✅

Content Preview:
### 1.4 Document Control

| Version | Date | Author | Changes |
|---------|------|--------|---------|
| 1.0 | 2025-05-01 | Engineering Team | Initial version |
| 1.1 | 2025-05-14 | Tech Architecture Council | Updated diagrams and monitoring section |

## 2. System Architecture


RESULT 2
-------------------------------
Returned Department : engineering
User Role           : Engineering
Allowed Departments : ['engineering', 'general']
Access Granted      : TRUE ✅

Content Preview:
#### 9.2.3 Next-Generation Infrastructure
* **Serverless Architecture**:
  * Function-as-a-Service for suitable workloads
  * Event-driven processing
  * Pay-per-use cost model
* **Zero-Downtime Opera

In [57]:
role_based_search_results(
    "company policies and salary structure",
    "general"
)


USER QUERY : company policies and salary structure
USER ROLE  : general
❌ Invalid role.


In [58]:
role_based_search_results(
    "finance hr engineering marketing general",
    "C-Level"
)


USER QUERY : finance hr engineering marketing general
USER ROLE  : C-Level

✅ C-LEVEL ACCESS — ALL DEPARTMENTS

RESULT 1
-------------------------------
Department : finance
User Role  : C-Level
Access     : TRUE ✅

Content Preview:
- **Revenue**: $2.6 billion, up 35% YoY, fueled by holiday campaigns and enterprise client acquisitions.
- **Gross Margin**: 64%, reflecting optimized pricing and operational efficiencies.
- **Operating Income**: $650 million, supported by strong revenue and cost discipline.
- **Net Income**: $325 m

RESULT 2
-------------------------------
Department : hr
User Role  : C-Level
Access     : TRUE ✅

Content Preview:
FINEMP1003  Krishna Malhotra     Business Analyst          Business  krishna.malhotra@fintechco.com      Pune    1984-12-20      2018-06-10 FINEMP1003  519865.26             12             5           84.34                   1       2024-07-24
 FINEMP1004     Aadhya Saxena    Marketing Manager      

RESULT 3
-------------------------------
Depart

In [59]:
role_based_search_results(
    query="company policies and employee handbook",
    user_role="Employees"
)


USER QUERY : company policies and employee handbook
USER ROLE  : Employees
QUERY DEPARTMENT: general

RESULT 1
-------------------------------
Returned Department : general
User Role           : Employees
Allowed Departments : ['general']
Access Granted      : TRUE ✅

Content Preview:
# Employee Handbook


RESULT 2
-------------------------------
Returned Department : general
User Role           : Employees
Allowed Departments : ['general']
Access Granted      : TRUE ✅

Content Preview:
## Table of Contents
1. [Welcome & Introduction](#welcome--introduction)
2. [Employee Onboarding & Benefits](#employee-onboarding--benefits)
3. [Leave Policies](#leave-policies)
4. [Work Hours & Attendance](#work-hours--attendance)
5. [Code of Conduct & Workplace Behavior](#code-of-conduct--workplac


RESULT 3
-------------------------------
Returned Department : general
User Role           : Employees
Allowed Departments : ['general']
Access Granted      : TRUE ✅

Content Preview:
### Dress Code & Offi

In [60]:
role_based_search_results(
    query="company policies and employee handbook",
    user_role="CR"
)


USER QUERY : company policies and employee handbook
USER ROLE  : CR
❌ Invalid role.
